# 24 - Does disappearing inventory later show native Sold?

This is a small prospective website-status experiment. The sample was frozen from **September 11 and 12 inventory before its outcome checks**. Run All only reads retained files. It never requests a page, creates a database or writes an export.

Read the cells in order: establish comparable inventory, inspect pending flags and exits, understand the frozen sample, then inspect observed outcomes. Native `Sold` is the measured endpoint. Completed deliveries, returns, transaction prices and national sales remain outside this experiment.
**Routine review:** 20 for inventory/prices, then 24 for the selected study; use 30
for quarterly assumptions. First-time learning is **00 -> 10 -> 11 -> 20 -> 24 -> 30**.
The default is the preserved **32-VIN September 12 study**, drawn from the original
seven-query Tesla Model 3 population. It is not the broader 101-query panel.

Edit `STUDY`, `AS_OF`, `EXTRA_CYCLE_PATHS` and `EXAMPLE_VIN` in Ordinary settings
below. Choosing another existing frozen study changes this review; it does not
freeze a replacement or modify either study. `AS_OF` is a historical evidence
cutoff. The actual observation dates, study paths and scope appear in the first tables.

| Output | Read it as |
| --- | --- |
| `scope_health`, `inventory_flow` | Which dates/population are comparable, and the stock-flow identity. |
| `frozen_sample`, `sampling` | Saved members and each arm's selected/full-frame denominator. |
| `outcome_counts`, `missing_outcome_bounds` | Observed native endpoints plus unresolved cases, not transaction accuracy. |
| `observation_windows`, `study_capacity` | Cutoff replay of fixed deadlines/capacity; not current permission to visit. |
| `example_source` through `example_classification` | One VIN from retained raw fields to the study result. |

A study not yet available at the cutoff yields no outcome tests. Unvisited,
Unavailable, failed or late observations remain unresolved; empty is not zero sales.
A missing file or changed hash blocks review so its path/evidence can be inspected.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p if (p/'src/vehicle_tracker').is_dir() else p/'vehicle'
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/vehicle_tracker').is_dir() or (p/'vehicle/src/vehicle_tracker').is_dir())
sys.path.insert(0, str(ROOT/'src'))
from vehicle_tracker.cycles import cycle_evidence, read_cycle_history
from vehicle_tracker.research_inputs import load_detail_evidence
from vehicle_tracker.status_experiment import (inventory_frame, read_experiment, score_plan,
                                               hypothesis_tests, fisher_greater, arm_outcomes, study_feasibility)



## Ordinary settings: choose an existing study and evidence cutoff

The default study stays selected until you explicitly change `STUDY`. Add later
same-scope cycle reports only for follow-up; the original comparison/sample stays fixed.
Effort values below are planning assumptions, not measured productivity.


In [ ]:
AS_OF = '2026-09-12T12:26:04.107393+00:00'
STUDY = ROOT / 'data/experiments/sales_method_20260912/study/plan.json'
EXTRA_CYCLE_PATHS = []
EXAMPLE_VIN = '5YJ3E1EA2PF398331'
MINUTES_PER_CHECK = 1.5
OPERATOR_MINUTES_PER_DAY = 20


### Advanced test/export overrides - normally unchanged

These existing hooks keep automated cutoff replay separate from ordinary editing.
`STUDY_OVERRIDE` may explicitly select another retained frozen study. No study is created here.


In [ ]:
AS_OF = globals().get('AS_OF_OVERRIDE', AS_OF)
STUDY = Path(globals().get('STUDY_OVERRIDE', STUDY))
EXTRA_CYCLE_PATHS = globals().get('EXTRA_CYCLE_PATHS_OVERRIDE', EXTRA_CYCLE_PATHS)
EXAMPLE_VIN = globals().get('EXAMPLE_VIN_OVERRIDE', EXAMPLE_VIN)
FOUR_MODEL = STUDY.parent.parent / 'four_model_inventory/cycle.json'
TZ = 'America/New_York'
print('Selected frozen study:', STUDY)
print('Evidence cutoff:', AS_OF, '| fixed-cutoff historical replay')
print('Extra follow-up cycles:', EXTRA_CYCLE_PATHS)


## 1. Is the evidence available and comparable?

The frozen manifest binds the exact plan and retained inventory files by hash. A changed file raises an error. Before the plan's preparation time the experiment is unavailable, and no hypothesis tests are shown.

The default study population is seven Tesla Model 3 year queries, ZIP 08542.
For another selected study, inspect its actual scope and retained dates below. Any four-model collection below is a **separate baseline**; its cars are never added to the Tesla experiment. A complete query means the requested scope reconciles, not national coverage.

In [ ]:
input_paths = {STUDY, STUDY.parent/'manifest.json'}
plan = read_experiment(STUDY, as_of=AS_OF)
research_tables = {'availability': pd.DataFrame([dict(as_of=AS_OF,
    status='available' if plan is not None else 'study not yet available')])}
display(research_tables['availability'])
if plan is not None:
    manifest = json.loads((STUDY.parent/'manifest.json').read_text(encoding='utf-8'))
    cycle_paths = [Path(p) for p in manifest['cycles']]
    source_days, source_inventory = read_cycle_history(cycle_paths, as_of=AS_OF)
    before_cycle_id = plan['pages'][0]['before_cycle_id']
    after_cycle_id = plan['pages'][0]['after_cycle_id']
    frozen_ids = [before_cycle_id, after_cycle_id]
    days = source_days.loc[source_days.cycle_id.isin(frozen_ids)].copy()
    inventory = source_inventory.loc[source_inventory.cycle_id.isin(frozen_ids)].copy()
    frame = inventory_frame(days, inventory, as_of=AS_OF)
    followup_days, followup_inventory = source_days, source_inventory
    if EXTRA_CYCLE_PATHS:
        followup_paths = list(dict.fromkeys(p.resolve() for p in [*cycle_paths, *map(Path, EXTRA_CYCLE_PATHS)]))
        followup_days, followup_inventory = read_cycle_history(followup_paths, as_of=AS_OF)
        if not followup_days.scope_id.eq(days.scope_id.iloc[0]).all():
            raise ValueError('Follow-up inventory must use the frozen study scope')
        input_paths.update(followup_paths)
        input_paths.update(Path(p) for p in followup_inventory.source_path)
        input_paths.update(p for path in followup_paths for p in path.parent.rglob('*')
                           if p.is_file() and p.suffix in {'.json', '.bin'})
    evidence = load_detail_evidence(ROOT, AS_OF)
    input_paths.update(map(Path, manifest['input_hashes']))
    input_paths.update(evidence['input_paths'])
    input_paths.update(cycle_paths)
    research_tables['scope_health'] = days
    research_tables['followup_inventory_health'] = followup_days
    query_frames = []
    for path in cycle_paths:
        state, coverage, reports = cycle_evidence(path, as_of=AS_OF)
        query_frames.append(coverage.assign(cycle_date=state['cycle_date']))
        input_paths.update(reports)
    research_tables['query_health'] = pd.concat(query_frames, ignore_index=True)
    print('Frozen inventory sources:', *cycle_paths, sep='\n')
    print('Selected scope:', days.scope_id.unique().tolist(), '| frozen VINs:', len(plan['pages']))
    display(days[['cycle_date', 'coverage_complete', 'coverage_reason', 'window_start', 'window_end']])
    if FOUR_MODEL.is_file():
        four_days, four_rows = read_cycle_history([FOUR_MODEL], as_of=AS_OF)
        input_paths.add(FOUR_MODEL)
        input_paths.update(p for p in FOUR_MODEL.parent.rglob('*')
                           if p.is_file() and p.suffix in {'.json', '.bin'})
        research_tables['four_model_health'] = four_days
        if not four_days.empty:
            four_counts = four_rows.groupby(['make', 'parent_model', 'year'], dropna=False).agg(
                observed_vins=('vin', 'nunique'), pending_flags_true=('purchase_pending', 'sum')).reset_index()
            four_counts['coverage_complete'] = bool(four_days.coverage_complete.all())
            research_tables['four_model_baseline'] = four_counts
            print('Separate four-model baseline; no flow comparison or pooled estimate:')
            display(four_counts)

## 2. Reconcile the inventory movement

Count entries and exits by retailer/VIN between the two complete dates. The identity is **beginning inventory + entries - exits = ending inventory**. It is inventory arithmetic: exits have not yet become sales. The whole sweep takes time, so its endpoints are observation windows rather than one instantaneous census.

In [ ]:
if plan is not None:
    ordered = days.set_index('cycle_id').loc[frozen_ids]
    before = inventory.loc[inventory.cycle_id.eq(before_cycle_id)]
    after = inventory.loc[inventory.cycle_id.eq(after_cycle_id)]
    old_vins = set(zip(before.retailer, before.vin))
    new_vins = set(zip(after.retailer, after.vin))
    flow = pd.DataFrame([dict(before_date=ordered.iloc[0].cycle_date,
        after_date=ordered.iloc[1].cycle_date, beginning=len(old_vins),
        entries=len(new_vins-old_vins), exits=len(old_vins-new_vins), ending=len(new_vins))])
    flow['reconciliation_difference'] = flow.beginning + flow.entries - flow.exits - flow.ending
    research_tables['inventory_flow'] = flow
    display(flow)

## 3. Was pending inventory more likely to disappear?

Pending is the **beginning-inventory flag** (September 11 in the default study), including for cars still listed at the ending date. This avoids comparing yesterday's exits with today's different pending population. Inspect the full 2x2 table before the p-value.

The one-sided Fisher calculation below is exploratory and was chosen after inspecting these inventories. It asks whether baseline pending and disappearance are positively associated under an independent-VIN sampling model. It is not a sale probability, a causal result, or one of the three prospective tests. Model/year composition and shared daily conditions can violate the model's assumptions.

In [ ]:
if plan is not None:
    pending_exit = pd.crosstab(frame.baseline_pending, frame.exited).reindex(
        index=[True, False], columns=[True, False], fill_value=0)
    pending_exit.index.name = 'baseline_pending'
    pending_exit.columns = ['exited', 'still_listed']
    association = pending_exit.copy()
    association['observed_exit_share'] = association.exited / association.sum(axis=1)
    counts = [int(n) for n in pending_exit.to_numpy().ravel()]
    exploratory = pd.DataFrame([dict(test='Baseline pending versus inventory exit',
        one_sided_p=fisher_greater(*counts), status='retrospective exploratory association',
        interpretation='Independence model; not sales conversion')])
    research_tables['pending_exit_table'] = association.reset_index()
    research_tables['exploratory_inventory_test'] = exploratory
    display(association)
    display(exploratory)

## 4. Inspect the frozen sample

There are four arms: pending exits, non-pending exits, still-listed pending controls, and still-listed non-pending controls. For the default September 12 study, selection requested ten VINs per arm. Only two eligible non-pending exits existed, so that frozen sample has **32 VINs**, not 40. Their shortage is kept; no replacement car is chosen after outcomes arrive.

`selection_probability` is the selected count divided by that arm's eligible population. In the default study, ten of 47 pending exits and ten of 477 non-pending controls have different sampling fractions. Do not pool the arms' raw Sold percentages into a population estimate. `full_frame` remains available for inspecting every beginning-inventory VIN and exclusion.

In [ ]:
if plan is not None:
    full_frame = pd.DataFrame(plan['frame'])
    selected = pd.DataFrame(plan['pages'])
    identity = ['retailer', 'vin', 'listing_id', 'arm', 'exclusion_reason']
    pd.testing.assert_frame_equal(full_frame[identity].sort_values('vin').reset_index(drop=True),
                                  frame[identity].sort_values('vin').reset_index(drop=True), check_dtype=False)
    sampling = full_frame.groupby('arm', dropna=False).agg(
        population=('vin', 'size'), selected=('selected_for_check', 'sum'),
        selection_probability=('selection_probability', 'first')).reset_index()
    research_tables.update(full_frame=full_frame, frozen_sample=selected, sampling=sampling)
    display(sampling)
    display(selected[['vin', 'listing_id', 'arm', 'selection_probability', 'last_asking_price_usd']])

## 5. Review outcomes before testing

Each VIN uses its first identity-matched native check in the fixed 48-hour primary window. `Available` and `Sold` resolve the binary endpoint. Other native values, including `Unavailable`, remain visible as unresolved; a later Sold check does not replace that fixed first outcome. Missing, failed and late checks also stay visible. A first-check Sold label corroborates that website endpoint; it does not invent a prior native Available observation.

The three planned positive-association comparisons are: pending exits versus pending controls; non-pending exits versus non-pending controls; and pending exits versus non-pending exits. P-values remain unavailable while the primary window is open or the required comparison has unresolved outcomes. The family uses three Bonferroni-adjusted tests. The two-car non-pending exit arm is especially imprecise.
### Classification rules to inspect

All native rules require an exact retailer/VIN/listing match and a valid capture clock. Native fields and interpreted purchase availability remain separate columns.

| Observation / rule | Concrete example | Classification and limit |
| --- | --- | --- |
| Complete comparable consecutive sweeps; VIN present before, absent after | September 11/12: 49 Tesla VIN exits | Inventory exit; sale unconfirmed |
| Partial/missing sweep or changed scope | Missing September 10 operating date | Withhold daily exits and matched-price movement |
| Native `Sold`, no conflicting purchase UI | VIN `5YJ3E1EA1MF994994`, listing `4620536` | Website Sold endpoint; transaction and exact timing unknown |
| Native `Available` + `Purchasable` + Get Started | VIN `5YJ3E1EA1MF053528`, listing `4650707` | Native Available, interpreted available |
| Native `Available` + Purchase in progress; `Purchasable` or `Reservable` | VIN `5YJ3E1EA0TF123048`, listing `4717673`, `Reservable/WIP` | Native Available, interpreted pending; no sale |
| Native `Available/Purchasable` + exact `On Hold` countdown | Synthetic rule example: `On Hold\n03:21` | Temporary pending only; arbitrary hold prose is not enough |
| Native `Available/NotPurchasable` | VIN `5YJ3E1EA9LF805054`, listing `4520264` | Native Available endpoint, interpreted unavailable; reason unknown |
| Native `Unavailable/NotPurchasable`, no conflicting UI | VIN `5YJ3E1EA2PF398331`, listing `4727313` | Matched native evidence; binary endpoint unresolved |
| Preorder/inspection wording without supported purchasable UI | Synthetic rule example: `Available/Reservable` + Pre-order | Native Available if parser matches; purchase availability remains unknown |
| Identity mismatch, contradictory UI, challenge or missing native saleStatus | Synthetic rule example: expected VIN A, returned VIN B | Reject/unresolved; never infer Sold or Available |
| Repeated Sold, same listing and no intervening Available | Same native Sold checked twice | Persistence, not two sales |
| First matched check in fixed wave | First `Unavailable`, later `Sold` (synthetic sequence) | First wave outcome remains unresolved; preserve later evidence separately |
| Sold then Available / exited VIN seen again | Synthetic sequence: Sold on day 1, Available on day 8 | Website reversal/reappearance; customer return unconfirmed |
| Matched-VIN asking price change | Retained VIN `5YJ3E1EA0RF763544`: $34,990 to $34,990 | $0 asking-price change; realized transaction price unknown |

The table gives the supported common cases; the parser also rejects unknown or conflicting combinations. Inspect `observed_status` alongside the untouched `saleStatus` and `purchaseType` before interpreting a result.


The arm table counts **distinct selected VINs**, separately showing attempted, identity-matched, Sold, Available, Unavailable, failed, unresolved starts and unvisited. Failed means at least one failed capture and no matched endpoint; unresolved-start counts can overlap a failed VIN. Attempts and matched captures are operational coverage, not transaction validation.

`Sold / (Sold + Available)` uses only resolved first endpoints. It can be biased when unresolved vehicles differ. The displayed **95% Wilson interval** describes that resolved subset under an independent Bernoulli model within an arm. Shared day/model conditions and selective missingness can invalidate that model; the interval does not correct them or describe quarterly sales accuracy. For a fully enumerated arm, it is model-based uncertainty, not uncertainty about a sampled fraction of that finite arm.

Alongside it, full-selected missing-outcome bounds are **Sold / selected** to **(Sold + unresolved) / selected**. These keep Unavailable, failed, unfinished and unvisited binary outcomes in the denominator. They are logical sensitivity bounds for this frozen website endpoint, not a confidence interval or transaction-sales bound. Repeated checks of one VIN never add independent vehicles. No pooled or national Sold estimate follows from these arm tables.

Attempted means the physical browser start occurred inside the wave, when that retained start is known. A start before the deadline with a capture or failure after it remains attempted, with `late_completion` and `late_identity_matched` shown separately; it does not resolve the fixed endpoint. A started visit without an available completion stays unresolved. A capture and its reservation count as one attempt, and repeated attempts still count as one attempted VIN in the arm table. Legacy captures without a recorded start use their physical check time.


In [ ]:
if plan is not None:
    outcomes = score_plan(plan, evidence['records'], followup_days, followup_inventory,
                          as_of=AS_OF, browser_health=evidence['browser_health'])
    tests = hypothesis_tests(outcomes, plan, as_of=AS_OF)
    # The shared helper validates that no selected VIN was filtered or duplicated.
    outcome_counts = arm_outcomes(outcomes, plan, as_of=AS_OF)
    repeat_counts = arm_outcomes(outcomes, plan, as_of=AS_OF, wave='repeat')
    # Keep the denominator arithmetic visible next to the study assumptions.
    resolved = outcome_counts.sold + outcome_counts.available
    assert outcome_counts.resolved.eq(resolved).all()
    assert outcome_counts.unresolved.eq(outcome_counts.selected - resolved).all()
    bounds = outcome_counts[['arm', 'selected', 'sold', 'unresolved']].copy()
    bounds['lower'] = bounds.sold / bounds.selected.where(bounds.selected.gt(0))
    bounds['upper'] = (bounds.sold + bounds.unresolved) / bounds.selected.where(bounds.selected.gt(0))
    research_tables.update(primary_outcomes=outcomes, outcome_counts=outcome_counts,
                           repeat_outcome_counts=repeat_counts, missing_outcome_bounds=bounds,
                           hypothesis_tests=tests, browser_health=evidence['browser_health'])
    display(outcome_counts)
    display(bounds)
    display(tests)
    display(outcomes[['vin', 'arm', 'primary_outcome', 'primary_native_status', 'primary_status', 'primary_checked_at', 'primary_attempts', 'primary_late_completions']])

## 6. Repeat the same VINs seven to nine days after selection

These are exact **selection-date windows**, not seven days after each car's first visit. Start is inclusive; end is exclusive. A check outside a window does not fill that endpoint. The timestamps below are calculated from the frozen plan, not rescheduled by rerunning this notebook.

Compare Sold persistence, Sold-to-Available changes and inventory reappearances. These are website reversals; labeling them customer returns would require independent evidence. `hours_exit_to_sold_observation` measures elapsed time between observations, not actual order processing or delivery time. Missing native Available history leaves a transition boundary unknown.

The capacity table below uses the **analysis cutoff** and those same frozen windows. `required_checks` excludes completed identity-matched first endpoints, including Unavailable. `available_capacity` counts the remaining selected checks that fit; `shortfall` stays missing if it cannot be collected in time. The scheduler includes retained starts, failures, unresolved reservations, rolling-day limits, per-VIN spacing and assumed time to finish each capture. An unresolved reservation has no assumed recovery time.

The **12 starts per rolling 24 hours is a local pilot setting**, not an established Carvana limit. Planning assumes 1.5 active minutes per check, at most 20 minutes of operator effort per rolling day, availability throughout the windows and no future competing work or retries. These are explicit assumptions to replace with measured effort. Capacity is not a guaranteed outcome or a reservation of future starts. Future `freeze` commands reject shortfalls and save the feasibility snapshot; they never resize this existing study. For current admission and deadline status, run the report command without `--as-of`; this notebook remains a reproducible fixed-cutoff review.


In [ ]:
if plan is not None:
    windows = pd.DataFrame([dict(wave=wave, start_utc=plan[wave+'_start'], end_utc=plan[wave+'_end'])
                            for wave in ['primary', 'repeat']])
    for field in ['start', 'end']:
        windows[field+'_new_york'] = pd.to_datetime(windows[field+'_utc'], utc=True).dt.tz_convert(TZ)
    analysis_cutoff = pd.Timestamp(AS_OF)
    windows['state'] = ['not started' if analysis_cutoff < pd.Timestamp(row.start_utc) else
                        'open' if analysis_cutoff < pd.Timestamp(row.end_utc) else 'closed' for row in windows.itertuples()]
    research_tables['observation_windows'] = windows
    research_tables['repeat_outcomes'] = outcomes[['vin', 'arm', 'repeat_status', 'repeat_outcome',
        'repeat_checked_at', 'first_sold_at', 'last_available_before_sold_at',
        'hours_exit_to_sold_observation', 'inventory_reappeared_at',
        'sold_to_available_between_waves', 'available_to_sold_between_waves']]
    display(windows)
    display(research_tables['repeat_outcomes'])
    feasibility = study_feasibility(plan, evidence['records'],
        browser_root=ROOT/'data/experiments/carvana_detail_batches', now=AS_OF,
        minutes_per_check=MINUTES_PER_CHECK, operator_minutes_per_day=OPERATOR_MINUTES_PER_DAY)
    research_tables['study_capacity'] = pd.DataFrame(feasibility['table'])
    research_tables['capacity_schedule'] = pd.DataFrame(feasibility['checks'])
    research_tables['browser_budget_at_cutoff'] = pd.DataFrame([feasibility['workload']])
    display(research_tables['study_capacity'])
    display(research_tables['browser_budget_at_cutoff'])
    display(repeat_counts)

## 7. Follow one VIN from source to conclusion

Change `EXAMPLE_VIN` in settings to another frozen sample member. The first table reads the actual retained search JSON; the next shows normalized observations; the last shows separately captured native detail evidence. Inspect their source paths and clocks together. A VIN matches the vehicle; its listing ID identifies the particular Carvana listing.

In [ ]:
if plan is not None:
    own = selected.loc[selected.vin.eq(EXAMPLE_VIN)]
    if own.empty:
        print('Choose EXAMPLE_VIN from the frozen_sample table.')
    else:
        source = Path(own.iloc[0].before_source)
        capture = json.loads(source.read_text(encoding='utf-8'))
        raw_vehicle = pd.json_normalize([v for v in capture['vehicles'] if v.get('vin') == EXAMPLE_VIN])
        raw_vehicle = raw_vehicle.reindex(columns=['vin', 'vehicleId', 'isPurchasePending',
            'vehiclePurchaseType', 'vehicleInventoryType', 'price.total'])
        raw_vehicle['source_path'] = str(source)
        raw_vehicle['observed_at'] = capture['captured_at_utc']
        observed = followup_inventory.loc[followup_inventory.vin.eq(EXAMPLE_VIN), ['vin', 'listing_id', 'observed_at_utc',
            'purchase_pending', 'asking_price_usd', 'source_path']]
        native = evidence['records']
        native = native.loc[native.vin.eq(EXAMPLE_VIN)] if not native.empty else native
        native = native.reindex(columns=['vin', 'listing_id', 'checked_at', 'available_at', 'saleStatus',
            'purchaseType', 'observed_status', 'parse_outcome', 'source'])
        input_paths.add(source)
        classification = outcomes.loc[outcomes.vin.eq(EXAMPLE_VIN)]
        research_tables.update(example_source=raw_vehicle, example_inventory=observed, example_native=native,
                               example_classification=classification)
        display(raw_vehicle)
        display(observed)
        display(native)
        display(classification[['vin', 'arm', 'primary_native_status', 'primary_status', 'primary_outcome']])

## 8. The next collection action is separate

`browser_health` reports retained browser attempts, not the current connection state. Browser navigation requires connected Chrome. Use the existing [browser batch guide](../docs/browser_detail_batches.md); do not substitute an anonymous HTTP route after an access block.

Read-only report from the repository root:

```powershell
.\.venv\Scripts\python.exe -B vehicle/scripts/run_status_experiment.py report --study vehicle/data/experiments/sales_method_20260912/study/plan.json
```

When the displayed window is open and further visits are intended, prepare a fresh bounded batch using `run_status_experiment.py batch --study <the plan above> --wave primary --limit 12 --destination <fresh direct child of vehicle/data/experiments/carvana_detail_batches>`. Use `--wave repeat` only during the frozen repeat window. The command keeps the selected VINs, checks prior observations/reservations and enforces the existing physical-check spacing. Then use `run_carvana_details.py next/record/fail/status` with that batch; no command here is executed by Run All.

For later inventory observations, add same-scope cycle paths to `EXTRA_CYCLE_PATHS` in this notebook's settings, or use `--cycles` with the report command. Those observations feed follow-up scoring; the frozen inventory pair, flow calculation and selected VINs stay fixed. Without later inventory, reappearance remains unobserved. The separate four-model baseline needs another complete, comparable day before a flow can be calculated.
The current conservative limit is **12 started browser visits per rolling 24 hours across all saved batches**, plus 15 seconds after each saved visit and 24 hours between checks of a VIN. Failures and unfinished reservations consume starts. This is browser-assisted work, not unattended collection. Existing historical evidence is retained even when it exceeds the newly introduced aggregate limit. The frozen 32-VIN study will retain any unfinished outcomes at its original deadline; do not extend the window or divide runs to force completion. Future default studies request three VINs per arm.

Future-study preview and freeze use `--minutes-per-check` and `--operator-minutes-per-day` to state effort assumptions. Review both primary and repeat required/capacity/shortfall rows before freezing. A rejected future plan creates no study files; existing frozen membership and windows are unchanged. Unvisited checks after a deadline stay missing.


In [ ]:
# These tables are the export contract. Inspect them directly in Jupyter first.
display(pd.DataFrame([dict(table=name, rows=len(table), columns=len(table.columns))
                      for name, table in research_tables.items()]))
print('Evidence cutoff:', AS_OF)
print('Run All completed offline; export requires the separate export_sales_proxy.py --notebook 24 command.')